# Notebook 04: Camada Silver - Tratamento + Qualidade de Dados

**Objetivo**
- Transformar os dados da camada Bronze em um conjunto **confiável e consistente**.
- Realizar análises de **qualidade de dados** por atributo (nulos, domínios, duplicados).
- Enriquecer a semântica para facilitar análises (ex.: `diagnosis_desc`).

**Saída**
- Tabela Delta: `silver.breast_cancer`

In [0]:
df_bronze = spark.table("bronze.breast_cancer")
display(df_bronze.limit(10))

mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,diagnosis,ingestion_timestamp
17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,0,2025-12-20T19:41:52.164Z
20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.0186,0.0134,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.186,0.275,0.08902,0,2025-12-20T19:41:52.164Z
19.69,21.25,130.0,1203.0,0.1096,0.1599,0.1974,0.1279,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.00615,0.04006,0.03832,0.02058,0.0225,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.243,0.3613,0.08758,0,2025-12-20T19:41:52.164Z
11.42,20.38,77.58,386.1,0.1425,0.2839,0.2414,0.1052,0.2597,0.09744,0.4956,1.156,3.445,27.23,0.00911,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.5,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.173,0,2025-12-20T19:41:52.164Z
20.29,14.34,135.1,1297.0,0.1003,0.1328,0.198,0.1043,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.01149,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.2,1575.0,0.1374,0.205,0.4,0.1625,0.2364,0.07678,0,2025-12-20T19:41:52.164Z
12.45,15.7,82.57,477.1,0.1278,0.17,0.1578,0.08089,0.2087,0.07613,0.3345,0.8902,2.217,27.19,0.00751,0.03345,0.03672,0.01137,0.02165,0.005082,15.47,23.75,103.4,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.1244,0,2025-12-20T19:41:52.164Z
18.25,19.98,119.6,1040.0,0.09463,0.109,0.1127,0.074,0.1794,0.05742,0.4467,0.7732,3.18,53.91,0.004314,0.01382,0.02254,0.01039,0.01369,0.002179,22.88,27.66,153.2,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368,0,2025-12-20T19:41:52.164Z
13.71,20.83,90.2,577.9,0.1189,0.1645,0.09366,0.05985,0.2196,0.07451,0.5835,1.377,3.856,50.96,0.008805,0.03029,0.02488,0.01448,0.01486,0.005412,17.06,28.14,110.6,897.0,0.1654,0.3682,0.2678,0.1556,0.3196,0.1151,0,2025-12-20T19:41:52.164Z
13.0,21.82,87.5,519.8,0.1273,0.1932,0.1859,0.09353,0.235,0.07389,0.3063,1.002,2.406,24.32,0.005731,0.03502,0.03553,0.01226,0.02143,0.003749,15.49,30.73,106.2,739.3,0.1703,0.5401,0.539,0.206,0.4378,0.1072,0,2025-12-20T19:41:52.164Z
12.46,24.04,83.97,475.9,0.1186,0.2396,0.2273,0.08543,0.203,0.08243,0.2976,1.599,2.039,23.94,0.007149,0.07217,0.07743,0.01432,0.01789,0.01008,15.09,40.68,97.65,711.4,0.1853,1.058,1.105,0.221,0.4366,0.2075,0,2025-12-20T19:41:52.164Z


## Enriquecimento: `diagnosis_desc`

A coluna `diagnosis` é numérica (0/1). Para facilitar leitura e análise,
criamos uma coluna descritiva:

- 0 → Benigno
- 1 → Maligno

In [0]:
from pyspark.sql.functions import when, col

df_silver = (
    df_bronze
    .withColumn(
        "diagnosis_desc",
        when(col("diagnosis") == 0, "Benigno")
        .when(col("diagnosis") == 1, "Maligno")
        .otherwise("Desconhecido")
    )
)

display(df_silver.select("diagnosis", "diagnosis_desc").limit(10))

diagnosis,diagnosis_desc
0,Benigno
0,Benigno
0,Benigno
0,Benigno
0,Benigno
0,Benigno
0,Benigno
0,Benigno
0,Benigno
0,Benigno


## Qualidade de Dados – Checks

Nesta seção verificamos:
1. Valores nulos por coluna
2. Duplicidades de registros
3. Domínio esperado de `diagnosis` (apenas 0 ou 1)
4. Estatísticas básicas (min/max/mean) para atributos numéricos

In [0]:
from pyspark.sql.functions import sum as _sum

nulls = df_silver.select([
    _sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns
])

display(nulls)

mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,diagnosis,ingestion_timestamp,diagnosis_desc
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
total_rows = df_silver.count()
distinct_rows = df_silver.dropDuplicates().count()

print("Total de linhas:", total_rows)
print("Total de linhas distintas:", distinct_rows)
print("Linhas duplicadas:", total_rows - distinct_rows)

Total de linhas: 569
Total de linhas distintas: 569
Linhas duplicadas: 0


In [0]:
display(
    df_silver.groupBy("diagnosis", "diagnosis_desc")
             .count()
             .orderBy("diagnosis")
)

diagnosis,diagnosis_desc,count
0,Benigno,212
1,Maligno,357


In [0]:
numeric_cols = [c for c, t in df_silver.dtypes if t in ("double", "int", "bigint")]

display(df_silver.select(numeric_cols).describe())

summary,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,diagnosis
count,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569,569
mean,14.127291739894563,19.28964850615117,91.96903339191566,654.8891036906857,0.096360281195079,0.10434098418277686,0.08879931581722322,0.048919145869947236,0.181161862917399,0.06279760984182778,0.4051720562390161,1.2168534270650269,2.8660592267135288,40.33707908611603,0.007040978910369071,0.025478138840070306,0.031893716344463946,0.011796137082601056,0.020542298769771532,0.0037949038664323383,16.269189806678394,25.677223198594014,107.2612126537786,880.5831282952545,0.13236859402460469,0.25426504393673144,0.27218848330404205,0.11460622319859404,0.29007557117750454,0.08394581722319855,0.6274165202108963
stddev,3.524048826212078,4.301035768166949,24.2989810387549,351.9141291816527,0.014064128137673616,0.0528127579325122,0.0797198087078935,0.03880284485915359,0.027414281336035712,0.007060362795084459,0.2773127329861041,0.5516483926172023,2.021854554042107,45.49100551613178,0.003002517943839067,0.017908179325677377,0.030186060322988394,0.006170285174046866,0.008266371528798399,0.0026460709670891942,4.833241580469324,6.146257623038323,33.60254226903635,569.3569926699492,0.022832429404835458,0.15733648891374194,0.20862428060813235,0.0657323411959421,0.06186746753751869,0.01806126734889399,0.4839179564031686
min,6.981,9.71,43.79,143.5,0.05263,0.01938,0.0,0.0,0.106,0.04996,0.1115,0.3602,0.757,6.802,0.001713,0.002252,0.0,0.0,0.007882,8.948E-4,7.93,12.02,50.41,185.2,0.07117,0.02729,0.0,0.0,0.1565,0.05504,0
max,28.11,39.28,188.5,2501.0,0.1634,0.3454,0.4268,0.2012,0.304,0.09744,2.873,4.885,21.98,542.2,0.03113,0.1354,0.396,0.05279,0.07895,0.02984,36.04,49.54,251.2,4254.0,0.2226,1.058,1.252,0.291,0.6638,0.2075,1


## Conclusões da Análise de Qualidade dos Dados

Com base nas verificações realizadas na etapa de qualidade de dados, foi possível concluir que o conjunto de dados apresenta **nível adequado de qualidade** para uso analítico. As principais conclusões são:

- **Completude:**  
  Não foram identificados valores nulos em nenhuma das colunas do dataset, garantindo que todas as observações possuem informações completas para análise.

- **Unicidade:**  
  A verificação de duplicidades indicou que não há registros duplicados, assegurando que cada linha representa uma observação única.

- **Consistência de Domínio:**  
  A coluna `diagnosis` apresentou exclusivamente os valores esperados (`0` para Benigno e `1` para Maligno), confirmando a consistência do domínio da variável alvo.

- **Sanidade dos Valores Numéricos:**  
  As estatísticas descritivas (mínimo, máximo e média) demonstraram que não há valores negativos ou incoerentes nas variáveis numéricas, estando todas em conformidade com o significado físico esperado das medições.

Dessa forma, os dados atendem aos critérios de **completude, unicidade e consistência**, estando aptos para serem persistidos na camada Silver e utilizados nas análises subsequentes.


## Persistência na Camada Silver

Após validações e enriquecimento, persistimos o dataset tratado em Delta Lake.

Tabela de saída:
- `silver.breast_cancer`

In [0]:
df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver.breast_cancer")

display(spark.table("silver.breast_cancer").limit(10))

mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,diagnosis,ingestion_timestamp,diagnosis_desc
17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,0,2025-12-20T19:41:52.164Z,Benigno
20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.0186,0.0134,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.186,0.275,0.08902,0,2025-12-20T19:41:52.164Z,Benigno
19.69,21.25,130.0,1203.0,0.1096,0.1599,0.1974,0.1279,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.00615,0.04006,0.03832,0.02058,0.0225,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.243,0.3613,0.08758,0,2025-12-20T19:41:52.164Z,Benigno
11.42,20.38,77.58,386.1,0.1425,0.2839,0.2414,0.1052,0.2597,0.09744,0.4956,1.156,3.445,27.23,0.00911,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.5,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.173,0,2025-12-20T19:41:52.164Z,Benigno
20.29,14.34,135.1,1297.0,0.1003,0.1328,0.198,0.1043,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.01149,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.2,1575.0,0.1374,0.205,0.4,0.1625,0.2364,0.07678,0,2025-12-20T19:41:52.164Z,Benigno
12.45,15.7,82.57,477.1,0.1278,0.17,0.1578,0.08089,0.2087,0.07613,0.3345,0.8902,2.217,27.19,0.00751,0.03345,0.03672,0.01137,0.02165,0.005082,15.47,23.75,103.4,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.1244,0,2025-12-20T19:41:52.164Z,Benigno
18.25,19.98,119.6,1040.0,0.09463,0.109,0.1127,0.074,0.1794,0.05742,0.4467,0.7732,3.18,53.91,0.004314,0.01382,0.02254,0.01039,0.01369,0.002179,22.88,27.66,153.2,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368,0,2025-12-20T19:41:52.164Z,Benigno
13.71,20.83,90.2,577.9,0.1189,0.1645,0.09366,0.05985,0.2196,0.07451,0.5835,1.377,3.856,50.96,0.008805,0.03029,0.02488,0.01448,0.01486,0.005412,17.06,28.14,110.6,897.0,0.1654,0.3682,0.2678,0.1556,0.3196,0.1151,0,2025-12-20T19:41:52.164Z,Benigno
13.0,21.82,87.5,519.8,0.1273,0.1932,0.1859,0.09353,0.235,0.07389,0.3063,1.002,2.406,24.32,0.005731,0.03502,0.03553,0.01226,0.02143,0.003749,15.49,30.73,106.2,739.3,0.1703,0.5401,0.539,0.206,0.4378,0.1072,0,2025-12-20T19:41:52.164Z,Benigno
12.46,24.04,83.97,475.9,0.1186,0.2396,0.2273,0.08543,0.203,0.08243,0.2976,1.599,2.039,23.94,0.007149,0.07217,0.07743,0.01432,0.01789,0.01008,15.09,40.68,97.65,711.4,0.1853,1.058,1.105,0.221,0.4366,0.2075,0,2025-12-20T19:41:52.164Z,Benigno


### Resumo da Camada Silver

- Dados lidos da camada Bronze
- Enriquecimento com `diagnosis_desc`
- Checks de qualidade executados:
  - nulos por coluna
  - duplicidades
  - domínio de `diagnosis`
  - estatísticas descritivas
- Persistência em `silver.breast_cancer`